# Q3 — CLIP ViT-B/16 zero-shot evaluation

Compute zero-shot per-class precision and recall using CLIP (ViT-B/16).

In [ ]:
# Install note: if CLIP is not available, install via pip:
# pip install ftfy regex tqdm
# pip install git+https://github.com/openai/CLIP.git

from pathlib import Path
import numpy as np
from PIL import Image
import torch
import clip
from torchvision import transforms
from sklearn.metrics import precision_recall_fscore_support

DATA_ROOT = Path('Datasets/dataset') if Path('Datasets/dataset').exists() else Path('Datasets/dataset2/images')
if not DATA_ROOT.exists(): raise RuntimeError('Dataset not found')
CLASSES = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

def build_image_list(root, classes):
    items = []
    for idx, c in enumerate(classes):
        p = Path(root)/c
        imgs = sorted([x for x in p.iterdir() if x.suffix.lower() in ['.jpg','.jpeg','.png']])
        for im in imgs:
            items.append((str(im), idx))
    return items

items = build_image_list(DATA_ROOT, CLASSES)
print('Total images for zero-shot:', len(items))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load('ViT-B/16', device=device)

# Prepare text prompts
prompts = [f'a photo of a {c}' for c in CLASSES]
text_tokens = clip.tokenize(prompts).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

# Evaluate images in batches to avoid OOM
batch_size = 32
y_true, y_pred = [], []
for i in range(0, len(items), batch_size):
    batch = items[i:i+batch_size]
    imgs = [preprocess(Image.open(p).convert('RGB')) for p,_ in batch]
    imgs = torch.stack(imgs).to(device)
    with torch.no_grad():
        image_features = model.encode_image(imgs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        sims = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        preds = sims.argmax(dim=1).cpu().numpy()
    for (_, label), p in zip(batch, preds):
        y_true.append(label)
        y_pred.append(int(p))

import numpy as np
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(CLASSES))), zero_division=0)
import pandas as pd
df = pd.DataFrame({'class': CLASSES, 'precision': prec, 'recall': rec, 'f1': f1, 'support': sup})
display(df)
print('Zero-shot evaluation complete')